# Core NLP Part 2 — News Classification (ESG Controversies)

Interview-prep notebook. Progression:

1. Setup and a **simulated ESG news dataset** (no external data needed)
2. **NLTK** text preprocessing (tokenise, stop-words, stemming) and TF-IDF
3. **Unsupervised**: clustering news with TF-IDF + KMeans
4. **Supervised, statistical**: binary controversy classifier (Logistic Regression, Naive Bayes, SVM)
5. **Training pipeline**: train/validation/test split, cross-validation, **F1**, threshold tuning
6. **BERT**: setup, embeddings, clustering on embeddings, fine-tuning
7. Interview Q&A and pitfalls

Task framing: given a headline, predict `is_controversy` (1 = ESG controversy, 0 = not), and as an extension the ESG pillar (E / S / G).

In [1]:
import warnings
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_colwidth", 100)

## 1. Simulated ESG controversy news

Real labelled ESG data is proprietary, so we generate headlines from templates.

Design choices that make it realistic enough for practice:
- **Class imbalance** (~20% controversies) as in real news flow.
- **Hard negatives**: positive ESG news and "denies allegations" items share vocabulary with controversies, so keyword rules fail.
- **Label noise** (~4%) to mimic annotator disagreement, so F1 will not be a perfect 1.0.
- A second label, the ESG **pillar** (E/S/G), for multi-class and clustering evaluation.

In [2]:
companies = ["Acme Energy", "Borealis Mining", "Cobalt Retail", "Delta Pharma", "Everest Bank",
             "Fjord Shipping", "Granite Steel", "Helix Foods", "Iris Telecom", "Jade Motors"]

templates = {
    "E": [
        "{c} fined after toxic waste leak into river",
        "{c} accused of illegal deforestation near protected area",
        "{c} faces probe over excessive carbon emissions at plant",
        "{c} oil spill damages coastline, regulators open inquiry",
    ],
    "S": [
        "{c} sued over workplace discrimination and harassment claims",
        "{c} factory workers strike over unsafe conditions and unpaid wages",
        "{c} supplier linked to child labour allegations",
        "{c} data breach exposes millions of customer records",
    ],
    "G": [
        "{c} board investigated over accounting fraud",
        "{c} executives charged in bribery and corruption scandal",
        "{c} shareholders revolt over excessive CEO pay and weak oversight",
        "{c} whistleblower alleges insider trading by senior management",
    ],
    "None": [
        "{c} reports quarterly earnings above analyst expectations",
        "{c} announces new CEO and board appointments",
        "{c} launches new product line in European markets",
        "{c} wins award for cutting emissions and improving worker safety",
        "{c} praised for strong governance and diversity programmes",
        "{c} denies allegations, regulator finds no wrongdoing",
        "{c} invests in renewable energy and community education projects",
        "{c} shares rise as board approves dividend and buyback",
    ],
}
suffixes = ["", ", analysts said", ", according to reports", " on Tuesday", ", sources say", " amid market volatility"]

rng = np.random.default_rng(SEED)
n = 1500
pillars = rng.choice(["E", "S", "G", "None"], size=n, p=[0.07, 0.07, 0.06, 0.80])

rows = []
for p in pillars:
    text = rng.choice(templates[p]).format(c=rng.choice(companies)) + rng.choice(suffixes)
    rows.append((text, p))

news = pd.DataFrame(rows, columns=["headline", "pillar"])
news["is_controversy"] = (news["pillar"] != "None").astype(int)

# Label noise: flip 4% of binary labels (annotator disagreement)
flip = rng.random(n) < 0.04
news.loc[flip, "is_controversy"] = 1 - news.loc[flip, "is_controversy"]

print(news.shape)
print(news["is_controversy"].value_counts(normalize=True).rename("share"))
news.sample(6, random_state=SEED)

(1500, 3)
is_controversy
0    0.758
1    0.242
Name: share, dtype: float64


,headline,pillar,is_controversy
1116,"Granite Steel factory workers strike over unsafe conditions and unpaid wages, according to reports",S,1
1368,Delta Pharma executives charged in bribery and corruption scandal,G,1
422,Jade Motors accused of illegal deforestation near protected area,E,1
413,"Delta Pharma announces new CEO and board appointments, analysts said",None,0
451,"Everest Bank wins award for cutting emissions and improving worker safety, sources say",None,0
861,"Everest Bank reports quarterly earnings above analyst expectations, sources say",None,0


## 2. Text preprocessing with NLTK and TF-IDF basics

Models need numbers, not strings. Classic pipeline:

| Step | What it does | Interview note |
|---|---|---|
| Lower-casing | `Fined` -> `fined` | Removes trivial vocabulary variants |
| Tokenisation | split text into words | `word_tokenize` handles punctuation better than `split()` |
| Stop-word removal | drop `the`, `of`, ... | **Keep negations** (`no`, `not`): "no wrongdoing" is not "wrongdoing" |
| Stemming / lemmatisation | `fined`, `fines` -> `fine` | Stemming is crude and fast (Porter); lemmatisation is dictionary-based |
| Vectorisation | tokens -> numbers | Bag-of-words counts, or **TF-IDF** |

**TF-IDF**: $\text{tfidf}(t,d) = \text{tf}(t,d)\cdot \log\frac{N}{\text{df}(t)}$ (scikit-learn adds smoothing). Words frequent in a document but rare across the corpus get high weight. Limitation: no word order or meaning; "bank fined" and "fined bank" are identical, and synonyms are unrelated. BERT fixes this.

In [3]:
!pip install -U nltk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 45.8 MB/s  0:00:00


In [4]:
import re

# NLTK is optional in this kernel: `pip install nltk`. Falls back to a regex tokeniser.
try:
    import nltk
    from nltk.corpus import stopwords
    from nltk.stem import PorterStemmer
    from nltk.tokenize import word_tokenize

    for pkg in ["punkt", "punkt_tab", "stopwords"]:
        nltk.download(pkg, quiet=True)
    STOP = set(stopwords.words("english")) - {"no", "not", "nor"}  # keep negations
    _stem = PorterStemmer().stem
    _tokenize = word_tokenize
    HAS_NLTK = True
except Exception:
    STOP = {"the", "a", "an", "of", "and", "in", "on", "at", "to", "as", "over", "after", "for", "by", "is"}
    _stem = lambda w: w
    _tokenize = lambda s: re.findall(r"[A-Za-z']+", s)
    HAS_NLTK = False

print("NLTK available:", HAS_NLTK)


def preprocess(text: str) -> str:
    tokens = [t for t in _tokenize(text.lower()) if t.isalpha() and t not in STOP]
    return " ".join(_stem(t) for t in tokens)


news["clean"] = news["headline"].map(preprocess)
news[["headline", "clean"]].head(5)

NLTK available: True


,headline,clean
0,Granite Steel invests in renewable energy and community education projects amid market volatility,granit steel invest renew energi commun educ project amid market volatil
1,"Helix Foods shares rise as board approves dividend and buyback, according to reports",helix food share rise board approv dividend buyback accord report
2,Everest Bank praised for strong governance and diversity programmes on Tuesday,everest bank prais strong govern divers programm tuesday
3,Delta Pharma invests in renewable energy and community education projects on Tuesday,delta pharma invest renew energi commun educ project tuesday
4,"Helix Foods sued over workplace discrimination and harassment claims, according to reports",helix food su workplac discrimin harass claim accord report


## 3. Unsupervised: clustering news (no labels)

**Idea**: embed each headline as a TF-IDF vector, then group similar vectors with **KMeans** (minimise within-cluster squared distance to centroids).

- Useful for **topic discovery** and for bootstrapping labels when none exist.
- KMeans needs `k` up front. Use the **silhouette score** (cohesion vs separation, -1..1) or the elbow plot, together with inspection of **top terms per cluster**.
- **Clusters are not classes.** KMeans finds the largest *textual* groups, not the ones you care about. With `k=4` the four ESG pillars do not come out as four clusters: all controversies share generic words and merge into one cluster, while frequent neutral templates get their own (ARI was negative in a first attempt). Fix: **over-cluster** (`k` larger than the number of classes), then name or merge clusters by inspecting top terms.
- Metrics against known labels: **purity** (share in the majority label per cluster; rises trivially with `k`, so it is only meaningful alongside a sensible `k`) and **Adjusted Rand Index** (1 = perfect, ~0 = random). ARI is low here by design: 20 clusters vs 4 pillars is a different granularity, so use purity or the per-cluster `share_controversy` table instead.
- TF-IDF vectors are L2-normalised, so Euclidean KMeans approximates cosine clustering.

In [5]:
from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import adjusted_rand_score, silhouette_score

tfidf = TfidfVectorizer(ngram_range=(1, 2), min_df=2)
X_all = tfidf.fit_transform(news["clean"])  # unsupervised: no labels used, so no label leakage

# News has many sub-topics per pillar, so k is larger than the 4 pillars
for k in [2, 4, 8, 12, 16, 20, 24]:
    km = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit(X_all)
    print(f"k={k:2d}  silhouette={silhouette_score(X_all, km.labels_):.3f}")

K = 20
km = KMeans(n_clusters=K, n_init=10, random_state=SEED).fit(X_all)
news["cluster"] = km.labels_

# Purity: share of items belonging to the majority label of their cluster
def purity(labels, clusters):
    return pd.crosstab(clusters, labels).max(axis=1).sum() / len(labels)

print(f"\nPurity vs pillar        : {purity(news['pillar'], news['cluster']):.3f}")
print(f"ARI vs pillar           : {adjusted_rand_score(news['pillar'], news['cluster']):.3f}")

terms = np.array(tfidf.get_feature_names_out())
order = km.cluster_centers_.argsort(axis=1)[:, ::-1]
summary = pd.DataFrame({
    "n": np.bincount(km.labels_),
    "top_terms": [", ".join(terms[order[c, :5]]) for c in range(K)],
    "share_controversy": news.groupby("cluster")["is_controversy"].mean().round(2),
    "main_pillar": news.groupby("cluster")["pillar"].agg(lambda s: s.value_counts().idxmax()),
})
summary.sort_values("share_controversy", ascending=False)

k= 2  silhouette=0.065
k= 4  silhouette=0.132
k= 8  silhouette=0.255
k=12  silhouette=0.304
k=16  silhouette=0.338
k=20  silhouette=0.332
k=24  silhouette=0.269

Purity vs pillar        : 1.000
ARI vs pillar           : 0.095


,n,top_terms,share_controversy,main_pillar
cluster,,,,
18,21,"coastlin, coastlin regul, damag, damag coastlin, spill damag",1.00,E
15,23,"supplier link, supplier, link child, child, child labour",1.00,S
8,27,"emiss plant, face probe, plant, probe excess, probe",1.00,E
19,47,"board investig, account, account fraud, fraud, investig account",0.98,G
6,31,"million, custom, expos million, record, million custom",0.97,S
14,35,"harass, claim, workplac discrimin, workplac, discrimin harass",0.97,S
9,25,"fine toxic, leak, wast, wast leak, leak river",0.96,E
17,24,"whistleblow, senior manag, insid trade, trade senior, trade",0.96,G
16,27,"execut, execut charg, briberi, briberi corrupt, corrupt scandal",0.96,G


## 4. Supervised setup: splitting the data

| Set | Purpose |
|---|---|
| **Train** | fit model parameters |
| **Validation** | choose model, hyper-parameters, decision threshold |
| **Test** | touched **once** at the end for an unbiased estimate |

Rules to state in an interview:
- **Stratify** on the label so the 20% positive rate is preserved in each split (essential under imbalance).
- **Fit vectorisers on train only** (put them inside a `Pipeline`) to avoid **data leakage** (IDF statistics leaking from val/test).
- For real news, prefer a **time-based split** (train on past, test on future); random splits overstate performance because the same story is reported many times (near-duplicates leak across splits).
- Deduplicate near-identical headlines before splitting.

In [6]:
from sklearn.model_selection import train_test_split

X = news["clean"]
y = news["is_controversy"]

# 60 / 20 / 20, stratified
X_train, X_tmp, y_train, y_tmp = train_test_split(X, y, test_size=0.4, stratify=y, random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_tmp, y_tmp, test_size=0.5, stratify=y_tmp, random_state=SEED)

for name, part in [("train", y_train), ("val", y_val), ("test", y_test)]:
    print(f"{name:5s} n={len(part):4d}  positive rate={part.mean():.3f}")

train n= 900  positive rate=0.242
val   n= 300  positive rate=0.243
test  n= 300  positive rate=0.240


## 5. Evaluation: why F1, not accuracy

With 80% negatives, predicting "no controversy" always gives 80% accuracy and is useless.

With confusion-matrix counts TP, FP, FN:

$$\text{Precision}=\frac{TP}{TP+FP},\qquad \text{Recall}=\frac{TP}{TP+FN},\qquad F_1=\frac{2PR}{P+R}$$

- **Precision**: of the headlines flagged, how many are real controversies (cost of false alarms for analysts).
- **Recall**: of real controversies, how many did we catch (cost of missing a scandal; usually dominant in ESG risk).
- **F1** is the harmonic mean, so it is low if either is low. Use $F_\beta$ with $\beta>1$ to weight recall more.
- Multi-class: **macro-F1** (unweighted mean over classes, good for rare classes), **micro-F1** (pooled counts), **weighted-F1** (by support).
- Also report **PR-AUC** for imbalanced problems; ROC-AUC can look optimistic.

In [7]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import Pipeline

# Baseline: always predicts the majority class -> high accuracy, zero F1
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
pred = dummy.predict(X_val)
print(f"Dummy   accuracy={accuracy_score(y_val, pred):.3f}  F1={f1_score(y_val, pred, zero_division=0):.3f}")

# TF-IDF + Logistic Regression inside a Pipeline (vectoriser fit on train only)
logreg = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced")),
]).fit(X_train, y_train)

pred = logreg.predict(X_val)
print(f"LogReg  accuracy={accuracy_score(y_val, pred):.3f}  F1={f1_score(y_val, pred):.3f}\n")
print(classification_report(y_val, pred, target_names=["no controversy", "controversy"]))
print("Confusion matrix [[TN FP] [FN TP]]:\n", confusion_matrix(y_val, pred))

Dummy   accuracy=0.757  F1=0.000
LogReg  accuracy=0.960  F1=0.913

                precision    recall  f1-score   support

no controversy       0.96      0.99      0.97       227
   controversy       0.97      0.86      0.91        73

      accuracy                           0.96       300
     macro avg       0.96      0.93      0.94       300
  weighted avg       0.96      0.96      0.96       300

Confusion matrix [[TN FP] [FN TP]]:
 [[225   2]
 [ 10  63]]


## 6. Statistical models, cross-validation and threshold tuning

Classic text classifiers on TF-IDF:
- **Multinomial Naive Bayes**: assumes words are conditionally independent given the class; very fast, strong small-data baseline.
- **Logistic Regression**: linear model of log-odds; calibrated probabilities, interpretable coefficients (L2 regularisation via `C`).
- **Linear SVM**: maximum-margin linear separator; often best on sparse high-dimensional text.

**Stratified K-fold CV** on the training set compares models with less variance than a single split. The decision **threshold** (default 0.5) is a hyper-parameter: tune it on validation to trade precision against recall, never on test.

In [8]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC

def make_pipe(clf):
    return Pipeline([("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)), ("clf", clf)])

candidates = {
    "NaiveBayes": make_pipe(MultinomialNB(alpha=0.5)),
    "LogReg": make_pipe(LogisticRegression(max_iter=1000, class_weight="balanced")),
    "LinearSVM": make_pipe(LinearSVC(class_weight="balanced")),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
cv_table = []
for name, pipe in candidates.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="f1")
    cv_table.append((name, scores.mean(), scores.std()))
pd.DataFrame(cv_table, columns=["model", "cv_f1_mean", "cv_f1_std"]).sort_values("cv_f1_mean", ascending=False)

,model,cv_f1_mean,cv_f1_std
0,NaiveBayes,0.905817,0.036535
1,LogReg,0.905817,0.036535
2,LinearSVM,0.871692,0.037712


In [9]:
from sklearn.metrics import average_precision_score, precision_recall_curve

best = candidates["LogReg"].fit(X_train, y_train)
val_proba = best.predict_proba(X_val)[:, 1]
print("Validation PR-AUC:", round(average_precision_score(y_val, val_proba), 3))

# Pick the threshold that maximises F1 on validation
prec, rec, thr = precision_recall_curve(y_val, val_proba)
f1s = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-12, None)
best_thr = thr[np.argmax(f1s)]
print(f"best threshold={best_thr:.3f}  val F1={f1s.max():.3f}  (default 0.5 F1={f1_score(y_val, val_proba >= 0.5):.3f})")

# Final, one-time evaluation on the untouched test set
test_proba = best.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= best_thr).astype(int)
print(f"\nTEST F1={f1_score(y_test, test_pred):.3f}")
print(classification_report(y_test, test_pred, target_names=["no controversy", "controversy"]))

Validation PR-AUC: 0.864
best threshold=0.841  val F1=0.913  (default 0.5 F1=0.913)

TEST F1=0.914
                precision    recall  f1-score   support

no controversy       0.97      0.98      0.97       228
   controversy       0.94      0.89      0.91        72

      accuracy                           0.96       300
     macro avg       0.95      0.94      0.94       300
  weighted avg       0.96      0.96      0.96       300



In [10]:
# Interpretability: strongest words for the controversy class
tfidf_fit, clf_fit = best.named_steps["tfidf"], best.named_steps["clf"]
vocab = np.array(tfidf_fit.get_feature_names_out())
coef = clf_fit.coef_[0]
print("Top controversy terms:", ", ".join(vocab[np.argsort(coef)[::-1][:10]]))
print("Top non-controversy terms:", ", ".join(vocab[np.argsort(coef)[:10]]))

# Extension: multi-class ESG pillar with macro-F1 (rare classes count equally)
Xp_train, Xp_test, yp_train, yp_test = train_test_split(
    news["clean"], news["pillar"], test_size=0.25, stratify=news["pillar"], random_state=SEED)
pillar_model = make_pipe(LinearSVC(class_weight="balanced")).fit(Xp_train, yp_train)
pp = pillar_model.predict(Xp_test)
print(f"\nPillar macro-F1={f1_score(yp_test, pp, average='macro'):.3f}")
print(classification_report(yp_test, pp))

Top controversy terms: excess, account, account fraud, investig account, investig, fraud, board investig, pharma invest, claim, harass claim
Top non-controversy terms: new, divers, prais strong, govern, prais, divers programm, govern divers, programm, strong govern, strong

Pillar macro-F1=1.000
              precision    recall  f1-score   support

           E       1.00      1.00      1.00        26
           G       1.00      1.00      1.00        25
        None       1.00      1.00      1.00       294
           S       1.00      1.00      1.00        30

    accuracy                           1.00       375
   macro avg       1.00      1.00      1.00       375
weighted avg       1.00      1.00      1.00       375



## 7. Towards BERT

**BERT** (Bidirectional Encoder Representations from Transformers) is a Transformer **encoder** pre-trained with masked-language-modelling on large corpora.

| | TF-IDF | BERT |
|---|---|---|
| Representation | sparse, word counts | dense, **contextual** (same word, different vector by context) |
| Word order / meaning | ignored | captured via self-attention |
| "Bank" (river vs finance), "no wrongdoing" | confused | handled in context |
| Cost | milliseconds, CPU | heavier; GPU helps for fine-tuning |

Three ways to use it, in increasing cost:
1. **Frozen embeddings** (mean-pooled hidden states) -> cluster or feed to Logistic Regression.
2. **Fine-tuning**: add a classification head on the `[CLS]` representation and train end-to-end with a small learning rate (2e-5 to 5e-5), few epochs.
3. **Domain models** such as **FinBERT** (financial text) when labelled finance data is limited.

Setup (run once in the `qf` environment; downloads weights on first use):

```bash
pip install torch transformers
```

Key objects: `AutoTokenizer` (WordPiece tokens, `[CLS]`/`[SEP]`, attention mask, max length 512) and `AutoModel` / `AutoModelForSequenceClassification`. We use `distilbert-base-uncased` (about 40% smaller, ~97% of BERT quality) so the demo is fast.

In [11]:
!pip install -U torch transformers

  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.6.1-py3-none-any.whl.metadata (6.8 kB)
  Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl.metadata (4.2 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.3/127.3 MB 106.4 MB/s  0:00:01eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 102.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 87.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 846.4/846.4 kB 50.8 MB/s  0:00:00
Using cached networkx-3.6.1-py3-none-any.whl (2.1 MB)
Using cached safetensors-0.8.0-cp310-abi3-macosx_11_0_arm64.whl (484 kB)
Using cached sympy-1.14.0-py3-none-any.whl (6.3 MB)
Using cached mpmath-1.3.0-py3-none-any.whl (536 kB)
  Attempting uninstall: idna0m╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/9 [networkx]
    Found existing installation: idna 3.16━━━━━━━━━━━━━━━━━━━━ 3/9 [networkx]
    Uninstalling idna-3.1

In [12]:
try:
    import torch
    from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer
    HAS_BERT = True
except ImportError:
    HAS_BERT = False

MODEL_NAME = "distilbert-base-uncased"
DEVICE = (
    "cuda" if HAS_BERT and torch.cuda.is_available()
    else "mps" if HAS_BERT and torch.backends.mps.is_available()
    else "cpu"
)
print("BERT stack available:", HAS_BERT, "| device:", DEVICE if HAS_BERT else "n/a")
if not HAS_BERT:
    print("Install with: pip install torch transformers  -> BERT cells below will be skipped.")

# BERT sees the raw headline (casing/negations/word order matter), not the stemmed text
raw_train, raw_val, raw_test = news.loc[X_train.index, "headline"], news.loc[X_val.index, "headline"], news.loc[X_test.index, "headline"]

if HAS_BERT:
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    enc = tok(["Acme Energy denies allegations, regulator finds no wrongdoing"], return_tensors="pt")
    print(tok.convert_ids_to_tokens(enc["input_ids"][0]))
    print("attention_mask:", enc["attention_mask"][0].tolist())

BERT stack available: True | device: mps


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

['[CLS]', 'ac', '##me', 'energy', 'denies', 'allegations', ',', 'regulator', 'finds', 'no', 'wrong', '##do', '##ing', '[SEP]']
attention_mask: [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]


In [13]:
def embed(texts, batch_size=64, max_length=64):
    """Mean-pooled DistilBERT embeddings (padding masked out)."""
    model = AutoModel.from_pretrained(MODEL_NAME).to(DEVICE).eval()
    texts = list(texts)
    out = []
    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = tok(texts[i:i + batch_size], padding=True, truncation=True,
                        max_length=max_length, return_tensors="pt").to(DEVICE)
            hidden = model(**batch).last_hidden_state
            mask = batch["attention_mask"].unsqueeze(-1)
            out.append(((hidden * mask).sum(1) / mask.sum(1)).cpu().numpy())
    return np.vstack(out)


if HAS_BERT:
    E_train, E_val, E_test = embed(raw_train), embed(raw_val), embed(raw_test)
    print("embedding shape:", E_train.shape)

    # (a) Unsupervised: cluster the embeddings and compare with the pillar labels
    from sklearn.preprocessing import normalize
    E_all = normalize(np.vstack([E_train, E_val, E_test]))
    all_idx = np.concatenate([X_train.index, X_val.index, X_test.index])
    km_bert = KMeans(n_clusters=4, n_init=10, random_state=SEED).fit(E_all)
    print("BERT-embedding KMeans ARI vs pillar:",
          round(adjusted_rand_score(news.loc[all_idx, "pillar"], km_bert.labels_), 3))

    # (b) Supervised on frozen embeddings
    lr_emb = LogisticRegression(max_iter=2000, class_weight="balanced").fit(E_train, y_train)
    print("Frozen-BERT + LogReg  val F1:", round(f1_score(y_val, lr_emb.predict(E_val)), 3))

W1015 21:52:09.946000 49805 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding shape: (900, 768)
BERT-embedding KMeans ARI vs pillar: 0.088
Frozen-BERT + LogReg  val F1: 0.906


In [15]:
FINE_TUNE = True  # set False to skip fine-tuning (a few minutes on CPU, faster on GPU/MPS)
EPOCHS, BATCH, LR = 2, 16, 3e-5

if HAS_BERT and FINE_TUNE:
    torch.manual_seed(SEED)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)

    # Weighted loss handles the 80/20 imbalance
    w = torch.tensor(len(y_train) / (2 * np.bincount(y_train)), dtype=torch.float, device=DEVICE)
    loss_fn = torch.nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

    def batches(texts, labels, size, shuffle):
        idx = np.random.permutation(len(texts)) if shuffle else np.arange(len(texts))
        for i in range(0, len(idx), size):
            sel = idx[i:i + size]
            enc = tok(list(texts.iloc[sel]), padding=True, truncation=True, max_length=64, return_tensors="pt")
            yield enc.to(DEVICE), torch.tensor(labels.iloc[sel].values, device=DEVICE)

    def predict(texts, labels):
        model.eval()
        preds = []
        with torch.no_grad():
            for enc, _ in batches(texts, labels, 64, False):
                preds.extend(model(**enc).logits.argmax(-1).cpu().tolist())
        return np.array(preds)

    for epoch in range(EPOCHS):
        model.train()
        for enc, labels in batches(raw_train, y_train, BATCH, True):
            opt.zero_grad()
            loss = loss_fn(model(**enc).logits, labels)
            loss.backward()
            opt.step()
        print(f"epoch {epoch + 1}: val F1 = {f1_score(y_val, predict(raw_val, y_val)):.3f}")

    print("Fine-tuned BERT TEST F1:", round(f1_score(y_test, predict(raw_test, y_test)), 3))
else:
    print("Fine-tuning skipped (needs transformers/torch and FINE_TUNE=True).")

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1: val F1 = 0.913
epoch 2: val F1 = 0.913
Fine-tuned BERT TEST F1: 0.922


## 8. Interview cheat-sheet

**Pipeline in one breath**: clean text -> TF-IDF (or BERT embeddings) -> stratified train/val/test -> cross-validate models on train -> tune threshold on validation -> report F1 / PR-AUC once on test -> error analysis.

**Common questions**

- *Why F1 over accuracy?* Imbalance: accuracy rewards predicting the majority class. F1 balances precision and recall on the positive class.
- *Precision vs recall for ESG controversies?* Missing a scandal (FN) is usually costlier, so favour recall (lower threshold, $F_2$), and accept analyst review of false alarms.
- *How do you handle imbalance?* `class_weight='balanced'`, resampling (on train only), threshold tuning, PR-AUC; avoid accuracy.
- *What is data leakage here?* Fitting TF-IDF on all data, near-duplicate stories across splits, random splits on time-ordered news, tuning on test.
- *Naive Bayes vs Logistic Regression?* NB is generative with an independence assumption, fast, good for tiny data; LR is discriminative, usually more accurate and gives better-calibrated probabilities.
- *Stemming vs lemmatisation?* Stemming chops suffixes (fast, can produce non-words); lemmatisation maps to dictionary lemma using POS (slower, cleaner).
- *Why can BERT beat TF-IDF?* Contextual embeddings capture word order, negation and synonymy ("no wrongdoing", "accused" vs "alleged").
- *When is TF-IDF + LR enough?* Small data, strong keyword signal, latency or cost limits, need for explainability; always keep it as the baseline BERT must beat.
- *How to choose k in KMeans?* Silhouette, elbow, stability across seeds, and business interpretability of top terms.
- *Why does BERT take 512 tokens max and how to handle long articles?* Quadratic attention cost; truncate, chunk and aggregate predictions, or use long-context models.
- *Production concerns*: drift (new scandals bring new vocabulary), periodic re-labelling, monitoring F1 on a sampled labelled stream, human-in-the-loop review, explainability for compliance.

**Caveats on this notebook**
- The data is templated and therefore easy; real headlines are noisier, so expect lower F1 and a larger BERT advantage.
- Binary F1 is capped near 0.91 by the 4% injected label noise. Pillar macro-F1 is a perfect 1.0 only because pillar labels were left clean; treat 1.0 as a warning sign on real data (leakage or trivial task).
- Odd terms such as "pharma invests" among the top controversy words are the model memorising noisy labels. Inspecting coefficients and misclassified rows is a quick way to catch this.
- NLTK, torch and transformers are not installed in the `qf` kernel; the notebook falls back to a regex tokeniser and skips the BERT cells until they are installed.